# Attach a Mouse Record & Load Dataset — Cloud Workstation

This notebook shows the recommended workflow for a **CodeOcean cloud workstation** session:

1. Load a `MouseRecord` from the dataset catalog.
2. Call `attach_mouse_record_to_workstation()` to attach all round and derived
   assets to the current workstation computation.
3. Build an `HCRDataset` from the now-mounted data.

In [15]:
from pathlib import Path

from aind_hcr_data_loader.codeocean_utils import (
    MouseRecord,
    attach_mouse_record,
    attach_mouse_record_to_workstation,
    print_attach_results,
)
from aind_hcr_data_loader.hcr_dataset import create_hcr_dataset_from_schema

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 — Configuration

Set the mouse ID and paths.  
`data_dir` is where CodeOcean mounts attached assets (`/root/capsule/data` by convention).

In [16]:
MOUSE_ID = "790322"
CATALOG_PATH = Path(f"/src/ophys-mfish-dataset-catalog/mice/{MOUSE_ID}.json")
DATA_DIR = Path("/root/capsule/data")

## 2 — Load the catalog record & attach assets

`MouseRecord.from_json_file()` parses the catalog JSON into a typed dataclass
with `.rounds` and `.derived_assets` dicts.

`attach_mouse_record_to_workstation()` then:
- searches CodeOcean for each asset by name,
- calls `client.computations.attach_data_assets()` with the current
  `CO_COMPUTATION_ID`, and
- returns one `AttachResult` per asset so you can inspect successes/failures.

> **Tip:** pass `dry_run=True` first to verify all assets resolve without
> actually attaching anything.

In [17]:
# Load the catalog record
record = MouseRecord.from_json_file(CATALOG_PATH)

print(f"Mouse:   {record.mouse_id}")
print(f"Rounds:  {list(record.rounds.keys())}")
print(f"Derived: {list(record.derived_assets.keys())}")

Mouse:   790322
Rounds:  ['R1', 'R2', 'R3', 'R4', 'R5']
Derived: ['cell_typing', 'pairwise_unmixing']


In [18]:
# Optional: dry-run first to confirm all assets are resolvable before attaching
dry_results = attach_mouse_record_to_workstation(record, dry_run=True)
print_attach_results(dry_results, dry_run=True)

  [rounds.R1]  HCR_790322_2025-11-05_13-00-00_processed_2025-11-10_20-36-20  →  ✓ found (dry run)
  [rounds.R2]  HCR_790322_2025-11-12_13-00-00_processed_2025-11-13_22-03-58  →  ✓ found (dry run)
  [rounds.R3]  HCR_790322_2025-11-19_13-00-00_processed_2025-11-21_01-31-13  →  ✓ found (dry run)
  [rounds.R4]  HCR_790322_2025-12-04_13-00-00_processed_2025-12-05_22-49-47  →  ✓ found (dry run)
  [rounds.R5]  HCR_790322_2025-12-11_13-00-00_processed_2025-12-12_23-47-08  →  ✓ found (dry run)
  [derived_assets.cell_typing]  HCR_790322_cell-typing_2026-03-11_12-00-00  →  ✓ found (dry run)
  [derived_assets.pairwise_unmixing]  HCR_790322_pairwise-unmixing_2026-03-11_12-00-00  →  ✓ found (dry run)


In [19]:
# Attach all round + derived assets to this workstation computation.
results = attach_mouse_record_to_workstation(record)
print_attach_results(results)

  [rounds.R1]  HCR_790322_2025-11-05_13-00-00_processed_2025-11-10_20-36-20  →  ✓ attached
  [rounds.R2]  HCR_790322_2025-11-12_13-00-00_processed_2025-11-13_22-03-58  →  ✓ attached
  [rounds.R3]  HCR_790322_2025-11-19_13-00-00_processed_2025-11-21_01-31-13  →  ✓ attached
  [rounds.R4]  HCR_790322_2025-12-04_13-00-00_processed_2025-12-05_22-49-47  →  ✓ attached
  [rounds.R5]  HCR_790322_2025-12-11_13-00-00_processed_2025-12-12_23-47-08  →  ✓ attached
  [derived_assets.cell_typing]  HCR_790322_cell-typing_2026-03-11_12-00-00  →  ✓ attached
  [derived_assets.pairwise_unmixing]  HCR_790322_pairwise-unmixing_2026-03-11_12-00-00  →  ✓ attached


## 3 — Build the HCRDataset

Now that the assets are mounted under `DATA_DIR`, pass the same catalog record
directly to `create_hcr_dataset_from_schema()`.  It discovers each round
directory automatically from the asset names in the record.

In [20]:
dataset = create_hcr_dataset_from_schema(CATALOG_PATH, DATA_DIR)
dataset.summary()

mouse_id: 790322
Could not load metadata for mouse 790322
Cell-typing asset attached: CellTypingFiles(asset='HCR_790322_cell-typing_2026-03-11_12-00-00', basic_results=✓, h5ad=✓)
HCR Dataset Summary
Mouse ID: 790322
Rounds: R1, R2, R3, R4, R5

Channels by round:
  R1: 405, 488, 561, 594
  R2: 405, 488, 561, 594, 638, 514
  R3: 514, 561, 594, 488, 638, 405
  R4: 561, 488, 638, 594, 405, 514
  R5: 638, 514, 594, 488, 561, 405

Cell-typing: ✓  (CellTypingFiles(asset='HCR_790322_cell-typing_2026-03-11_12-00-00', basic_results=✓, h5ad=✓))

Segmentation files by round:
  R1: resolutions 0, 2, centroids: ✓
  R2: resolutions 0, 2, centroids: ✗
  R3: resolutions 0, 2, centroids: ✗
  R4: resolutions 0, 2, centroids: ✗
  R5: resolutions 0, 2, centroids: ✗

Spot detection files by round:
  R1: channels 594, 488, 561
    594: spots ✓, stats files: 3
    488: spots ✓, stats files: 3
    561: spots ✓, stats files: 3
  R2: channels 561, 488, 594, 638, 514
    561: spots ✓, stats files: 5
    488: spot

## 4 — (Optional) Use the auto-dispatcher

If the same script needs to run in both a workstation *and* a regular capsule,
use `attach_mouse_record()` instead.  It inspects env vars at runtime and
routes to the correct attach function automatically:

| Env var present | Dispatches to |
|---|---|
| `CO_COMPUTATION_ID` | `attach_mouse_record_to_workstation` |
| `CO_CAPSULE_ID` | `attach_mouse_record_to_capsule` |
| neither (explicit `pipeline_id`) | `attach_mouse_record_to_pipeline` |

In [21]:
# Works in a workstation, capsule, or pipeline — no code changes needed.
results = attach_mouse_record(record)
print_attach_results(results)

  [rounds.R1]  HCR_790322_2025-11-05_13-00-00_processed_2025-11-10_20-36-20  →  ✓ attached
  [rounds.R2]  HCR_790322_2025-11-12_13-00-00_processed_2025-11-13_22-03-58  →  ✓ attached
  [rounds.R3]  HCR_790322_2025-11-19_13-00-00_processed_2025-11-21_01-31-13  →  ✓ attached
  [rounds.R4]  HCR_790322_2025-12-04_13-00-00_processed_2025-12-05_22-49-47  →  ✓ attached
  [rounds.R5]  HCR_790322_2025-12-11_13-00-00_processed_2025-12-12_23-47-08  →  ✓ attached
  [derived_assets.cell_typing]  HCR_790322_cell-typing_2026-03-11_12-00-00  →  ✓ attached
  [derived_assets.pairwise_unmixing]  HCR_790322_pairwise-unmixing_2026-03-11_12-00-00  →  ✓ attached


---
## Quick-start — copy this into your own notebook

```python
from pathlib import Path
from aind_hcr_data_loader.codeocean_utils import (
    MouseRecord,
    attach_mouse_record_to_workstation,
    print_attach_results,
)
from aind_hcr_data_loader.hcr_dataset import create_hcr_dataset_from_schema
from aind_hcr_data_loader.pairwise_dataset import create_pairwise_unmixing_dataset

# ── configuration ────────────────────────────────────────────────────────────
MOUSE_ID     = "782149"  # use a mouse that has a pairwise_unmixing derived asset
CATALOG_PATH = Path(f"/src/ophys-mfish-dataset-catalog/mice/{MOUSE_ID}.json")
DATA_DIR     = Path("/root/capsule/data")

# ── attach & load ────────────────────────────────────────────────────────────
record  = MouseRecord.from_json_file(CATALOG_PATH)
results = attach_mouse_record_to_workstation(record)
print_attach_results(results)

dataset = create_hcr_dataset_from_schema(CATALOG_PATH, DATA_DIR)
dataset.summary()

# ── pairwise unmixing (optional) ─────────────────────────────────────────────
# The pairwise asset name lives in derived_assets["pairwise_unmixing"] when present.
pairwise_asset_name = record.derived_assets.get("pairwise_unmixing")

if pairwise_asset_name is not None:
    pw_ds = create_pairwise_unmixing_dataset(
        mouse_id=MOUSE_ID,
        pairwise_asset_path=DATA_DIR / pairwise_asset_name,
        source_dataset=dataset,   # delegates zarr / segmentation calls
    )
    pw_ds.summary()
else:
    print("No pairwise_unmixing asset found in catalog record — skipping.")
```